# LLM 08. Finalize Topic Classification

12번/12.5/17번 노트북의 ML 자동분류 + GPT mini fallback 결과를 검증하고, 아직 final에 없는 `memo_id`만 최종 라벨로 확정합니다.

- 입력: `ml_classification_detail`
- 출력: `classification_detail_final` (`append_new_only`: 이미 확정된 memo_id는 유지)
- 원본 row 확장 Tableau 테이블은 마지막 단계인 15번 노트북에서 생성합니다.


In [ ]:
import sys
import importlib

from pyspark.sql import functions as F

PROJECT_ROOT = "/Workspace/Users/jungryo.lee@lge.com/prj_TV_voc"
SRC_ROOT = f"{PROJECT_ROOT}/src"

if SRC_ROOT not in sys.path:
    sys.path.append(SRC_ROOT)

import common.config_loader as config_loader
import ml.final_classification_builder as final_classification_builder

importlib.reload(config_loader)
importlib.reload(final_classification_builder)

from common.config_loader import load_config, get_output_table, get_reference_table, get_source_table
from ml.final_classification_builder import (
    build_final_classification_detail_df,
    save_final_classification_detail,
    summarize_ml_classification_result,
)

config = load_config(f"{PROJECT_ROOT}/config/settings_intellytics.yaml")

print("settings =", config["path"]["settings"])
print("source =", get_source_table(config, "raw_review_table"))
print("ml_detail =", get_output_table(config, "ml_classification_detail"))
print("final_detail =", get_output_table(config, "classification_detail_final"))


In [ ]:
# 1. 12번 결과 검증: unresolved_pending_fallback_rows가 0에 가까울수록 최종화 준비가 잘 된 상태입니다.
stage12_summary = summarize_ml_classification_result(spark, config)
stage12_summary


In [ ]:
# 2-3. 최종 detail 생성 및 저장
# 이미 classification_detail_final에 있는 memo_id는 재저장하지 않고, 신규 memo_id만 append합니다.
final_detail_df = build_final_classification_detail_df(
    spark,
    config,
)

final_detail_rows = final_detail_df.count()
final_detail_distinct_memo_ids = final_detail_df.select("memo_id").dropDuplicates().count()
low_volume_rule_rows = final_detail_df.where(F.col("classification_stage") == "low_volume_group_rule").count()

final_detail_table = save_final_classification_detail(
    spark,
    config,
    final_detail_df,
    write_mode="append_new_only",
)

{
    "final_detail_table": final_detail_table,
    "final_detail_rows": final_detail_rows,
    "final_detail_distinct_memo_ids": final_detail_distinct_memo_ids,
    "low_volume_rule_rows": low_volume_rule_rows,
}


In [ ]:
# 최종 detail topic 분포 확인
final_detail_table = get_output_table(config, "classification_detail_final")
category_mapping_table = get_reference_table(config, "category_mapping_table")

final_detail_saved_df = spark.table(final_detail_table).where(F.col("prompt_version") == config["version"]["prompt_version"])

display(
    final_detail_saved_df.alias("t")
    .join(
        spark.table(category_mapping_table).alias("m"),
        on=["cate_1_depth", "cate_2_depth"],
        how="left",
    )
    .groupBy(
        "t.cate_1_depth",
        "m.cate_1_depth_kor",
        "t.cate_2_depth",
        "m.cate_2_depth_kor",
        "t.sc_measurement",
        "t.pred_topic_type",
        "t.pred_topic",
    )
    .agg(
        F.count("*").alias("memo_id_cnt"),
        F.avg("confidence_score").alias("avg_confidence"),
    )
    .orderBy("t.cate_1_depth", "t.cate_2_depth", "t.sc_measurement", F.desc("memo_id_cnt"))
)


In [ ]:
# 최종 detail 샘플 확인
display(
    final_detail_saved_df.select(
        "cate_1_depth",
        "cate_2_depth",
        "sc_measurement",
        "memo_id",
        "memo",
        "pred_topic",
        "pred_topic_type",
        "classification_stage",
        "confidence_score",
        "llm_used_yn",
    )
    .orderBy("cate_1_depth", "cate_2_depth", "sc_measurement")
    .limit(100)
)
